# Your scraper has an accent

Two clients. Same machine, same address, same URL. One gets a block page and
the other gets the site.

The only difference between them is what their TLS handshake and HTTP/2
connection look like on the wire. No proxy, no JavaScript, no captcha solver.

In [1]:
import html
import json
import re

import httpx
from IPython.display import HTML, display

from pybrws.client import get

PROBE = "https://tls.peet.ws/api/all"
BLOCKED = (403, 503)

# Bright enough for a projector, in either notebook theme.
GOOD, BAD, WARN, DIM = "#2ea043", "#f85149", "#d29922", "#8b949e"
CELL = "padding:7px 18px 7px 0;white-space:nowrap;text-align:left;"


def fetch(client, url):
    """Return (status, body) from either client, or (None, error text)."""
    try:
        if client == "pybrws":
            response = get(url, follow_redirects=5, timeout=30)
            return response.status, response.text()
        with httpx.Client(http2=True, follow_redirects=True, timeout=30) as http:
            response = http.get(url)
        return response.status_code, response.text
    except Exception as error:
        return None, "%s: %s" % (type(error).__name__, error)


def page_title(page):
    found = re.search(r"<title[^>]*>(.*?)</title>", page, re.S | re.I)
    return html.escape(found.group(1).strip()[:70]) if found else "(no title)"


def show(label, status, page, height=360):
    """Summarise a result, then preview it in a sandboxed frame.

    The summary is plain markup so it survives a renderer that strips
    iframes; the preview is a bonus where frames are allowed.
    """
    served = status is not None and status not in BLOCKED and status != 429
    tone = GOOD if served else BAD
    display(HTML(
        '<div style="font:14px/1.6 system-ui;padding:10px 14px;border-radius:6px;'
        'border:1px solid %s"><b>%s</b>'
        '<span style="color:%s;font-weight:700;margin-left:12px">%s</span>'
        '<span style="color:%s;margin-left:12px">%s bytes</span>'
        '<div style="color:%s;margin-top:4px">&lt;title&gt; %s</div></div>'
        % (tone, html.escape(label), tone,
           "HTTP %s" % status if status else "failed",
           DIM, format(len(page), ","), DIM, page_title(page))
    ))
    display(HTML(
        '<iframe sandbox="" loading="lazy" style="width:100%%;height:%dpx;'
        'border:1px solid %s;border-radius:6px;background:#fff;margin-top:6px" '
        'srcdoc="%s"></iframe>' % (height, DIM, html.escape(page, quote=True))
    ))


def compare(url, height=360):
    for client in ("httpx", "pybrws"):
        status, page = fetch(client, url)
        show("%s  %s" % (client, url), status, page, height)


def table(header, rows):
    head = "".join('<th style="%s">%s</th>' % (CELL, h) for h in header)
    body = "".join("<tr>%s</tr>" % "".join(r) for r in rows)
    display(HTML(
        '<div style="overflow-x:auto"><table style="font:13px/1.4 ui-monospace,'
        'SFMono-Regular,Menlo,monospace;border-collapse:collapse">'
        '<tr style="border-bottom:1px solid %s;color:%s">%s</tr>%s</table></div>'
        % (DIM, DIM, head, body)
    ))


def td(text, colour=None, bold=False):
    return '<td style="%scolor:%s;%s">%s</td>' % (
        CELL, colour or "inherit", "font-weight:700" if bold else "", text)


def verdict(status):
    """Classify a status: served, blocked, or rate limited by our own traffic."""
    if status is None:
        return BAD, "&#10007; failed"
    if status == 429:
        return WARN, "&#8211; 429 rate limited"
    if status in BLOCKED:
        return BAD, "&#10007; %d" % status
    return GOOD, "&#10003; %d" % status


## 1. zillow.com

`httpx` is a mature HTTP/2 client over Python's standard TLS stack. `pybrws`
differs from it only in the bytes of its ClientHello, its HTTP/2 preamble and
its header order.

In [2]:
compare("https://www.zillow.com/")

C:\Users\epcos\labs\pybrws\.venv\Lib\site-packages\IPython\core\display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## 2. etsy.com, behind DataDome

A category page rather than the homepage. The strictest rules sit on the
endpoints most worth scraping: etsy refuses `/` and `/?q=` to this client too,
while serving `/c/jewelry`.

In [3]:
compare("https://www.etsy.com/c/jewelry")

## 3. A wider sweep

Four protection vendors, measured the same way. The rows where this does not
work are included: a target both clients fail is not a failure of the
measurement, and leaving it out would be choosing the result.

`429` is rate limiting earned by a day of measuring, not a verdict on the
client, so it is marked apart from a block.

In [4]:
TARGETS = [
    ("https://www.zillow.com/", "CloudFront"),
    ("https://www.etsy.com/c/jewelry", "DataDome"),
    ("https://www.yelp.com/", "DataDome"),
    ("https://www.homedepot.com/", "Akamai"),
    ("https://www.quora.com/", "Cloudflare"),
    ("https://www.etsy.com/", "DataDome, homepage"),
    ("https://www.g2.com/", "Cloudflare"),
    ("https://example.com/", "control, serves everyone"),
]



rows = []
for url, guard in TARGETS:
    cells = [td(html.escape(url)), td(guard, DIM)]
    for client in ("pybrws", "httpx"):
        colour, text = verdict(fetch(client, url)[0])
        cells.append(td(text, colour, bold=True))
    rows.append(cells)

table(["target", "protection", "pybrws", "httpx"], rows)

target,protection,pybrws,httpx
https://www.zillow.com/,CloudFront,✓ 200,✗ 403
https://www.etsy.com/c/jewelry,DataDome,✓ 200,✗ 403
https://www.yelp.com/,DataDome,✓ 200,✗ 403
https://www.homedepot.com/,Akamai,✓ 200,✗ 403
https://www.quora.com/,Cloudflare,✗ 403,✗ 403
https://www.etsy.com/,"DataDome, homepage",✗ 403,✗ 403
https://www.g2.com/,Cloudflare,✗ 403,✗ 403
https://example.com/,"control, serves everyone",✓ 200,✓ 200


## 4. Why the baseline arm is not optional

A query string gets both sites to serve a request they otherwise refuse. It is
tempting to read that as the client working. Running the plain client beside it
says otherwise on one of the two.

The same parameter, five interleaved rounds each.

In [5]:
PARAM_TARGETS = ["https://www.g2.com/?x=", "https://www.etsy.com/?x="]

rows = []
for url in PARAM_TARGETS:
    cells = [td(html.escape(url))]
    results = {}
    for client in ("pybrws", "httpx"):
        status = fetch(client, url)[0]
        results[client] = status
        colour, text = verdict(status)
        cells.append(td(text, colour, bold=True))
    served = lambda s: s is not None and s not in BLOCKED and s != 429
    if served(results["pybrws"]) and not served(results["httpx"]):
        reading, colour = "the fingerprint is what decides", GOOD
    elif served(results["pybrws"]) and served(results["httpx"]):
        reading, colour = "the query string alone does it, we add nothing", WARN
    else:
        reading, colour = "no difference", DIM
    cells.append(td(reading, colour))
    rows.append(cells)

table(["target", "pybrws", "httpx", "reading"], rows)

target,pybrws,httpx,reading
https://www.g2.com/?x=,✓ 200,✓ 200,"the query string alone does it, we add nothing"
https://www.etsy.com/?x=,✓ 200,✗ 403,the fingerprint is what decides


Without the second column, both rows look like a win. One of them is not.

Every claim in this project carries that arm, which is also why three findings
during its development were retracted rather than published.

## 5. What the server actually saw

`tls.peet.ws` reports back the fingerprint of whoever asked. The first row is
a real Firefox 157, captured by opening that URL in the browser.

In [6]:
with open("fixtures/reference-firefox-157-peet.json", encoding="utf-8") as handle:
    reference = json.load(handle)

with httpx.Client(http2=True, timeout=30) as http:
    seen_baseline = http.get(PROBE).json()
seen_ours = json.loads(get(PROBE, timeout=30).body)

target_ja4 = reference["tls"]["ja4"]
target_h2 = reference["http2"]["akamai_fingerprint"]

rows = []
for index, (name, seen) in enumerate(
    [("Firefox 157", reference), ("pybrws", seen_ours), ("httpx", seen_baseline)]
):
    cells = [td(name, bold=True)]
    for value, target in ((seen["tls"]["ja4"], target_ja4),
                          (seen["http2"]["akamai_fingerprint"], target_h2)):
        if index == 0:
            cells.append(td(html.escape(value), DIM))
        else:
            match = value == target
            mark = "&#10003; " if match else "&#10007; "
            cells.append(td(mark + html.escape(value), GOOD if match else BAD))
    rows.append(cells)

table(["client", "JA4", "HTTP/2 fingerprint"], rows)

client,JA4,HTTP/2 fingerprint
Firefox 157,t13d1517h2_8daaf6152771_3cbfd9057e0d,"1:65536;2:0;4:131072;5:16384|12517377|0|m,p,a,s"
pybrws,✓ t13d1517h2_8daaf6152771_3cbfd9057e0d,"✓ 1:65536;2:0;4:131072;5:16384|12517377|0|m,p,a,s"
httpx,✗ t13d1712h1_ab0a1bf427ad_8e6e362c5eac,"✗ 1:4096;2:0;4:65535;5:16384;3:100;6:65536|16777216|0|m,a,s,p"


## 6. Does a session change the outcome?

Both etsy and g2 refuse one endpoint while serving another. A browser never
arrives at a page cold: it has been around the site, it carries cookies, and
it reuses the connection it already opened.

So does reaching a refused endpoint *after* a served one change anything?

Four arms. The first three separate what a session changes, since it carries
cookies and reuses a connection at once:

| arm | cookies | connection |
|---|---|---|
| **cold** | none | new |
| **cookies only** | carried | new, the pool is dropped in between |
| **warm** | carried | reused |

The fourth is the one that matters for architecture: hand those cookies to
the plain client and see whether they still work. If they travel, a browser
can earn a token once and anything cheap can spend it.

In [7]:
from pybrws.session import Session

WARMUP, REFUSED = "https://www.etsy.com/c/jewelry", "https://www.etsy.com/"


def cold():
    return fetch("pybrws", REFUSED)[0]


def cookies_only():
    with Session() as session:
        session.get(WARMUP)
        session.close()      # keeps the jar, drops the pooled connection
        return session.get(REFUSED).status


def warm():
    with Session() as session:
        session.get(WARMUP)
        return session.get(REFUSED).status


def handed_to_httpx():
    """Earn the cookies here, spend them from the plain client."""
    with Session() as session:
        session.get(WARMUP)
        jar = {cookie.name: cookie.value for cookie in session.cookies}
    with httpx.Client(timeout=30, http2=True, follow_redirects=True,
                      cookies=jar) as http:
        return http.get(REFUSED).status_code, len(jar)


ARMS = [("pybrws, cold", cold), ("pybrws, cookies only", cookies_only),
        ("pybrws, warm", warm)]

rows = []
for label, run in ARMS:
    colour, text = verdict(run())
    rows.append([td(label), td(text, colour, bold=True), td("", DIM)])

status, carried = handed_to_httpx()
colour, text = verdict(status)
rows.append([td("httpx, given those cookies", BAD if colour is BAD else None),
             td(text, colour, bold=True),
             td("%d cookies carried over" % carried, DIM)])

table(["arm", "result", ""], rows)

arm,result,
"pybrws, cold",✗ 403,
"pybrws, cookies only",✓ 200,
"pybrws, warm",✓ 200,
"httpx, given those cookies",✗ 403,6 cookies carried over


One measurement is not a finding. These sites have already been seen to answer
the same request differently from one run to the next, so repeat anything
green here before believing it; `tools/ab_test.py` keeps a timestamped log.

Measured over four interleaved rounds, with no variance in any arm: the first
three come out `0/4`, `4/4`, `4/4`, and the fourth comes out `0/4`.

So the cookies do the work and the connection does not, and they are **not
transferable**. The same cookie values, sent by a client with a different
fingerprint, are refused every time. The token is bound to whoever earned it.

That rules out the cheapest architecture, where a browser collects a cookie
and anything at all spends it. What is left still works, but the client doing
the spending has to carry the fingerprint too, which is what this one is.

## 7. The part that is easy to oversell

The fingerprint is necessary and it is not sufficient. Crossing it against the
address it comes from, three rounds each:

| | pybrws | httpx |
|---|---|---|
| **home connection** | 3/3 served | 0/3 |
| **commercial ISP proxy** | 0/3 | 0/3 |

One cell of four passes. A proxy address sold as clean is itself a signal, and
here it scored worse than an ordinary domestic line.

And nothing here solves a JavaScript challenge. A site that issues one is out
of reach for any HTTP client, by construction.